In [8]:
import os
import sys
import google.generativeai as genai
from dotenv import load_dotenv
load_dotenv()
from collections import defaultdict
import json 
import torch

def find_root_path(path:str, word:str):
    parts = path.split(word, 1)    
    return parts[0] + word if len(parts) > 1 else path 

try:
    current_path = os.path.abspath(__file__)
except NameError:
    current_path = os.getcwd()
    
root_folder = find_root_path(os.path.abspath(current_path), 'art_lang')
sys.path.append(root_folder)

from rpod.decision_transformer.adapter import FrozenTextAdapter  
device =  "cuda" if torch.cuda.is_available() else "cpu"
print(device) 

%load_ext autoreload
%autoreload 2

cpu
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from rpod.dataset_generation.gpt_prompting import annotate_traj_behaviors, annotate_traj_behaviors2
api_key = os.getenv("OPENAI_API_KEY")
# api_key = os.getenv("GOOGLE_API_KEY")

ids = [i for _ in range(100) for i in range(7)]

# annotations = annotate_traj_behaviors(ids, api_key)
annotations = annotate_traj_behaviors2(ids, host="openai", api_key=api_key)

In [ ]:
# Aggregate descriptions by (id, command)
agg = defaultdict(list)
for entry in annotations.values():
    key = (entry["id"], entry["command"])
    agg[key].append(entry["description"])

# Convert to list of dicts for JSONL output
merged = [
    {"id": k[0], "command": k[1], "description": v}
    for k, v in agg.items()
]

# Save as JSONL
with open("summarized.jsonl", "w") as f:
    for m in merged:
        f.write(json.dumps(m) + "\n")

In [ ]:
# get an embeddings for each 

# collect embeddings from the text command 
MODEL = os.getenv("FTA_MODEL", "distilbert-base-uncased")   # this is encoder only     
adapter = FrozenTextAdapter(model_name=MODEL, out_dim=384, output_mode="tokens").to(device).eval()

with torch.inference_mode():
    out = adapter(commands)  # forward pass 

In [35]:
for k, v in annotations.items():
    print(f"ID: {v['id']}, Description: {v['description']}")

ID: 0, Description: Maintain safety by utilizing an offset approach arc.
ID: 1, Description: Executing a swift flyaround ensures safe clearance margins.
ID: 2, Description: Maintain position at -50 m V-bar for safety.
ID: 3, Description: Execute swift V-bar descent to maintain safety margins.
ID: 4, Description: A high-speed pass is executed without E/I separation.
ID: 5, Description: Execute gentle phasing pass within designated safety corridor.
ID: 6, Description: Initiating a safe retreat along the V-bar.
ID: 1, Description: Safety margins are maintained during aggressive flyaround maneuvers.
ID: 1, Description: Safety margins are maintained during fast circumnavigation.
ID: 1, Description: Prioritize timing for safe, tight clearance maneuvers.
ID: 1, Description: Safety margins are maintained during rapid circumvention.
ID: 0, Description: A safe offset arc is maintained during circling.
ID: 1, Description: Execute a rapid flyaround with tight clearance margins.
ID: 2, Description: